# EfficientNet-B0 촬영 각도 분류기: Google Colab 학습

런타임 유형을 **GPU**로 변경한 뒤 위에서부터 실행합니다. 이 노트북은 Colab 기본 PyTorch·torchvision을 사용하며 PyTorch 재설치를 하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'Drive의 프로젝트 경로를 확인하세요: {PROJECT_ROOT}'

In [ ]:
# 의존성 설치를 하지 않습니다. Colab 기본 런타임의 호환된 Torch 조합을 그대로 검증합니다.
import sys
try:
    import torch, torchvision
    from PIL import Image
except ImportError as exc:
    raise RuntimeError('Colab 기본 런타임에 torch, torchvision, Pillow가 필요합니다. 새 GPU 런타임으로 다시 연결하세요.') from exc
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('Python:', sys.version)
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess
os.chdir(PROJECT_ROOT)
# 빈 라벨, 오타, 누락 클래스가 있으면 이 단계가 중단됩니다. CSV를 수정한 뒤 다시 실행하세요.
subprocess.run([sys.executable, '-m', 'scripts.prepare_angle_classification_dataset', '--copy-mode', 'copy'], check=True)

In [ ]:
# -u와 Popen을 사용해 자식 프로세스의 진행률·지표를 실시간으로 표시합니다.
command = [
    sys.executable, '-u', '-m', 'scripts.train_efficientnet_b0_angle_classifier',
    '--epochs', '50', '--batch-size', '32', '--num-workers', '2', '--device', 'auto',
]
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
print('실행 명령:', ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait() != 0:
    raise RuntimeError('EfficientNet-B0 학습이 실패했습니다.')

In [ ]:
weights = PROJECT_ROOT / 'runs/efficientnet_b0_angle/best.pt'
subprocess.run([sys.executable, '-m', 'scripts.evaluate_efficientnet_b0_angle_classifier', '--weights', str(weights)], check=True)